[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/058.%20The%20ABC_XYZ%20Inventory%20Classification%20Problem/P58-Tier-7_executed.ipynb)

---


# 58. The ABC/XYZ Inventory Classification Problem

## Tier 7: Human–AI Collaboration

### Purpose, inputs, and outputs

This notebook demonstrates a simulated workflow in which an ensemble of AI-style classifiers assigns ABC/XYZ categories, a priority rule sends selected items to human review, and human decisions are combined with automated recommendations. The goal is to study the workflow and its outputs—not to train or validate a real machine-learning model.

The input is a collection of stock keeping units (SKUs). Each SKU has an annual value, at least two monthly demand observations, and optional business context. The output is a final category for each SKU, together with the ensemble recommendation, review decision when applicable, and descriptive workflow metrics. ABC classes represent value tiers; XYZ classes represent demand-variability tiers.

### Model assumptions and scope

- Annual value and demand coefficient of variation (CV) are the numerical inputs to the simulated classifiers. Business context and strategic importance influence review and expert decisions, but they are not measured ground truth.
- The ensemble consists of simulated model votes. A component's configured accuracy setting controls the chance that its vote is perturbed away from the rule-based base category. These settings are not estimates of real-world predictive accuracy.
- Human decisions and review durations are simulated. There is no independently labeled reference category in this dataset, so classification accuracy, override correctness, cost savings, stockout reduction, and learning gains cannot be measured.
- Expert profile accuracy settings contribute to the profile-selection score. They do not establish the correctness of a simulated human decision.
- The generated data are synthetic. The notebook does not model replenishment quantities, lead times, service levels, inventory carrying costs, or the financial consequences of a classification.
- Per-SKU processing time records measured Python execution time for classification and review logic. Human review duration is a separate simulated quantity; it is not elapsed time spent by the Python program.

### ABC/XYZ rules used by the simulated AI votes

For this example, the AI's base category uses the following thresholds:

| Dimension | Class | Rule |
|---|---|---|
| Annual value | A | Annual value $> 100{,}000$ |
| Annual value | B | $25{,}000 < \text{annual value} \leq 100{,}000$ |
| Annual value | C | Annual value $\leq 25{,}000$ |
| Demand CV | X | $\mathrm{CV} < 0.10$ |
| Demand CV | Y | $0.10 \leq \mathrm{CV} < 0.25$ |
| Demand CV | Z | $\mathrm{CV} \geq 0.25$ |

For monthly demands $d_{i1},\ldots,d_{iT}$ of SKU $i$, the sample coefficient of variation is

$$
\mathrm{CV}_i = \frac{s_i}{\bar{d}_i},
\qquad
\bar{d}_i = \frac{1}{T}\sum_{t=1}^{T} d_{it},
\qquad
s_i = \sqrt{\frac{1}{T-1}\sum_{t=1}^{T}(d_{it}-\bar{d}_i)^2}.
$$

Here, $T$ is the number of monthly observations, $\bar{d}_i$ is their mean, and $s_i$ is their sample standard deviation. The CV is dimensionless, so it describes relative rather than absolute variability. If mean demand is zero, the implementation treats CV as infinite and assigns the Z variability tier. The code uses NumPy's sample standard deviation setting (`ddof=1`).

### Collaborative workflow

The order of operations is important: the AI first classifies every SKU; a rule then determines which recommendations need review; the review queue is prioritized and limited by human capacity; finally, each reviewed item receives a human decision. Items that qualify for review but exceed capacity keep their AI recommendation and are reported as AI-only decisions. Within a priority level, the stable sort preserves the SKUs' input order.

For SKU $i$, let $p_i$ be the winning ensemble vote share, $u_i$ the number of uncertainty indicators, and $v_i$ its annual value. The review priority is assigned by the threshold rules in `determine_review_priority`: HIGH applies if $p_i < 0.6$, $v_i > 200{,}000$, the item is marked CRITICAL, or $u_i > 2$; otherwise MEDIUM applies if $p_i < 0.8$, $v_i > 50{,}000$, or $u_i > 0$; all remaining items are LOW priority. HIGH and MEDIUM items enter the review queue. The queue is sorted by priority, and at most the configured capacity is reviewed.

For model $m$ and category $c$, the ensemble adds model $m$'s weight when it votes for $c$. Let $s_{ic}$ denote the total weight assigned to category $c$ for SKU $i$. The winning category and its confidence score are

$$
\hat{c}_i = \underset{c}{\operatorname{argmax}}\; s_{ic},
\qquad
p_i = \frac{s_{i\hat{c}_i}}{\sum_c s_{ic}}.
$$

The candidate categories are AX, AY, AZ, BX, BY, BZ, CX, CY, and CZ. The implementation selects the first category in that listed order if scores tie. Thus $p_i$ is the fraction of total ensemble weight supporting the winning vote; it is not a calibrated probability of correctness.

The rule-based AI base category is determined from annual value and CV. Each simulated model either votes for that base category or, according to its configured perturbation setting, votes for a randomly selected alternative. Weighted voting then produces the ensemble recommendation. Human review similarly uses configured rules and random draws; it is a simulation, not an independent assessment of a known true category.

This is a threshold-based triage and review process, not a learned model or an optimization problem. If there are $n$ SKUs, $M$ ensemble members, $q$ items in the review queue, $r$ completed reviews, and $E$ expert profiles, the main work is approximately $O(nM + q\log q + rE)$, treating the profile text checks as bounded work. The terms correspond to generating votes, sorting the queue, and selecting a reviewer for each completed review.

### Interpreting the outputs

The useful outputs are category counts, review workload, acceptance and override counts, vote shares, trust scores, measured Python execution times, and simulated human review durations. They describe the behavior of this simulation under its stated parameters. They are not evidence of predictive accuracy or business impact. A target review percentage or expected productivity gain requires independent data and validation before it can be interpreted as a real outcome.


In [ ]:
from dataclasses import dataclass
from enum import Enum
import random
import time
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np

# Reproducible random perturbations and simulated human decisions.
np.random.seed(42)
random.seed(42)
plt.style.use("default")


In [ ]:
class DecisionStatus(Enum):
    """Possible outcomes of a simulated human review."""

    AI_RECOMMENDED = "AI_RECOMMENDED"
    HUMAN_ACCEPTED = "HUMAN_ACCEPTED"
    HUMAN_OVERRIDDEN = "HUMAN_OVERRIDDEN"
    PENDING_REVIEW = "PENDING_REVIEW"
    COLLABORATIVE_REFINED = "COLLABORATIVE_REFINED"


@dataclass
class SKU:
    """A stock keeping unit and the demand data used for classification."""

    sku_id: str
    annual_value: float
    monthly_demands: List[float]
    business_context: str = ""
    strategic_importance: str = ""
    market_trend: str = ""

    def __post_init__(self) -> None:
        if not np.isfinite(self.annual_value) or self.annual_value < 0:
            raise ValueError("annual_value must be finite and non-negative")
        if len(self.monthly_demands) < 2:
            raise ValueError("At least two monthly demand observations are required")
        if not np.all(np.isfinite(self.monthly_demands)):
            raise ValueError("monthly_demands must contain only finite values")
        if any(demand < 0 for demand in self.monthly_demands):
            raise ValueError("monthly_demands must be non-negative")

        self.avg_demand = float(np.mean(self.monthly_demands))
        self.std_demand = float(np.std(self.monthly_demands, ddof=1))
        self.cv = (
            self.std_demand / self.avg_demand
            if self.avg_demand > 0
            else float("inf")
        )


@dataclass
class AIClassification:
    """Simulated ensemble recommendation and its supporting information."""

    predicted_category: str
    confidence_score: float
    explanation: str
    feature_importance: Dict[str, float]
    uncertainty_factors: List[str]
    alternative_categories: Dict[str, float]


@dataclass
class HumanReview:
    """Simulated expert decision and review metadata."""

    reviewer_id: str
    review_timestamp: str
    decision_status: DecisionStatus
    final_category: str
    reasoning: str
    confidence_factors: List[str]
    business_considerations: List[str]
    review_duration_seconds: float


@dataclass
class CollaborativeResult:
    """Final classification and workflow information for one SKU."""

    sku: SKU
    ai_classification: AIClassification
    human_review: Optional[HumanReview]
    final_category: str
    collaboration_type: str
    total_processing_time: float
    trust_score: float


@dataclass
class HumanExpert:
    """A simulated expert profile used for review assignment."""

    expert_id: str
    name: str
    expertise_areas: List[str]
    experience_years: int
    accuracy_rate: float
    average_review_time: float
    trust_in_ai: float


@dataclass
class PartnershipMetrics:
    """Descriptive workflow metrics; no accuracy is inferred without labels."""

    total_skus_processed: int
    ai_only_count: int
    human_reviewed_count: int
    human_overridden_count: int
    collaborative_refined_count: int
    average_processing_time: float
    trust_development_score: float
    review_rate: float
    override_rate: float


print("Data structures for the simulated human-AI workflow are ready.")

In [ ]:
def create_ai_ensemble_classifier() -> Dict:
    """Return the configured simulated ensemble and review thresholds."""
    return {
        "models": [
            {"name": "Random Forest", "weight": 0.3, "accuracy": 0.85},
            {"name": "Gradient Boosting", "weight": 0.3, "accuracy": 0.87},
            {"name": "Neural Network", "weight": 0.2, "accuracy": 0.83},
            {"name": "Logistic Regression", "weight": 0.2, "accuracy": 0.82},
        ],
        "explainability_enabled": True,
        "confidence_threshold": 0.8,
        "uncertainty_detection": True,
    }


def ai_classify_sku(sku: SKU, ai_model: Dict) -> AIClassification:
    """Generate weighted, perturbed votes for one SKU.

    Component accuracy values are simulation parameters for whether a vote is
    perturbed. They do not measure or imply accuracy against a labeled truth.
    """
    categories = [
        "AX", "AY", "AZ", "BX", "BY", "BZ", "CX", "CY", "CZ"
    ]
    class_scores = {category: 0.0 for category in categories}

    if sku.annual_value > 100000:
        abc = "A"
    elif sku.annual_value > 25000:
        abc = "B"
    else:
        abc = "C"

    if sku.cv < 0.10:
        xyz = "X"
    elif sku.cv < 0.25:
        xyz = "Y"
    else:
        xyz = "Z"
    base_category = abc + xyz

    for model in ai_model["models"]:
        vote = base_category
        if np.random.random() > model["accuracy"]:
            alternatives = [category for category in categories if category != base_category]
            vote = str(np.random.choice(alternatives))
        class_scores[vote] += model["weight"]

    total_score = sum(class_scores.values())
    if total_score <= 0:
        raise ValueError("The ensemble must have a positive total model weight")
    class_scores = {
        category: score / total_score
        for category, score in class_scores.items()
    }
    predicted_category = max(class_scores, key=class_scores.get)
    confidence = class_scores[predicted_category]

    explanation = generate_ai_explanation(sku, predicted_category, confidence)
    raw_importance = {
        "annual_value": min(0.8, sku.annual_value / 200000),
        "demand_cv": min(0.6, sku.cv / 0.5) if np.isfinite(sku.cv) else 0.6,
        "demand_stability": max(0.1, 1.0 - sku.cv) if np.isfinite(sku.cv) else 0.1,
        "value_rank": min(0.4, sku.annual_value / 500000),
    }
    importance_total = sum(raw_importance.values())
    feature_importance = (
        {key: value / importance_total for key, value in raw_importance.items()}
        if importance_total > 0
        else {key: 0.0 for key in raw_importance}
    )

    uncertainty_factors = []
    if confidence < 0.7:
        uncertainty_factors.append("Low ensemble consensus")
    if sku.cv > 0.4:
        uncertainty_factors.append("High demand variability")
    if 20000 < sku.annual_value < 30000:
        uncertainty_factors.append("Near an annual-value threshold")
    if 0.08 < sku.cv < 0.12:
        uncertainty_factors.append("Near a demand-CV threshold")

    alternative_categories = {
        category: probability
        for category, probability in class_scores.items()
        if probability > 0.1 and category != predicted_category
    }

    return AIClassification(
        predicted_category=predicted_category,
        confidence_score=confidence,
        explanation=explanation,
        feature_importance=feature_importance,
        uncertainty_factors=uncertainty_factors,
        alternative_categories=alternative_categories,
    )


def generate_ai_explanation(sku: SKU, category: str, confidence: float) -> str:
    """Explain how the category relates to value and demand variability."""
    abc_class, xyz_class = category

    if abc_class == "A":
        abc_explanation = (
            f"Annual value of ${sku.annual_value:,.0f} places this SKU in the "
            "simulated A value tier"
        )
    elif abc_class == "B":
        abc_explanation = (
            f"Annual value of ${sku.annual_value:,.0f} places this SKU in the "
            "simulated B value tier"
        )
    else:
        abc_explanation = (
            f"Annual value of ${sku.annual_value:,.0f} places this SKU in the "
            "simulated C value tier"
        )

    if xyz_class == "X":
        xyz_explanation = f"Demand is relatively stable (CV={sku.cv:.3f})"
    elif xyz_class == "Y":
        xyz_explanation = f"Demand has moderate variability (CV={sku.cv:.3f})"
    else:
        xyz_explanation = f"Demand has high variability (CV={sku.cv:.3f})"

    confidence_explanation = (
        f"The ensemble's winning vote share is {confidence:.2f}; this is not a "
        "calibrated probability of correctness"
    )
    return f"{abc_explanation}. {xyz_explanation}. {confidence_explanation}."


print("Simulated ensemble classification and explanation functions are ready.")

In [ ]:
def create_human_experts() -> List[HumanExpert]:
    """Create the illustrative expert profiles used in the simulation."""
    return [
        HumanExpert(
            expert_id="EXP001",
            name="Dr. Sarah Chen",
            expertise_areas=["Electronics", "High-value items", "Seasonal products"],
            experience_years=12,
            accuracy_rate=0.94,
            average_review_time=45.0,
            trust_in_ai=0.78,
        ),
        HumanExpert(
            expert_id="EXP002",
            name="Michael Rodriguez",
            expertise_areas=["Consumer goods", "Fast-moving items", "Demand forecasting"],
            experience_years=8,
            accuracy_rate=0.91,
            average_review_time=38.0,
            trust_in_ai=0.82,
        ),
        HumanExpert(
            expert_id="EXP003",
            name="Prof. James Kim",
            expertise_areas=["Industrial equipment", "Spare parts", "Long-tail items"],
            experience_years=15,
            accuracy_rate=0.96,
            average_review_time=52.0,
            trust_in_ai=0.75,
        ),
        HumanExpert(
            expert_id="EXP004",
            name="Lisa Thompson",
            expertise_areas=["Fashion", "Trend analysis", "New product launches"],
            experience_years=6,
            accuracy_rate=0.88,
            average_review_time=35.0,
            trust_in_ai=0.85,
        ),
    ]


def determine_review_priority(sku: SKU, ai_classification: AIClassification) -> str:
    """Assign HIGH, MEDIUM, or LOW priority using the stated triage rules."""
    if (
        ai_classification.confidence_score < 0.6
        or sku.annual_value > 200000
        or sku.strategic_importance == "CRITICAL"
        or len(ai_classification.uncertainty_factors) > 2
    ):
        return "HIGH"

    if (
        ai_classification.confidence_score < 0.8
        or sku.annual_value > 50000
        or len(ai_classification.uncertainty_factors) > 0
    ):
        return "MEDIUM"

    return "LOW"


def select_expert_for_review(sku: SKU, experts: List[HumanExpert]) -> HumanExpert:
    """Choose the highest-scoring available profile for this SKU."""
    if not experts:
        raise ValueError("At least one expert is required for human review")

    expert_scores = []
    context = sku.business_context.casefold()
    for expert in experts:
        score = sum(
            2.0 for area in expert.expertise_areas
            if area.casefold() in context
        )
        score += expert.experience_years / 10.0
        score += expert.accuracy_rate * 2.0
        score += 1.0 / (1.0 + expert.average_review_time / 60.0)
        expert_scores.append((score, expert))

    return max(expert_scores, key=lambda item: item[0])[1]


print("Review triage and expert assignment functions are ready.")

In [ ]:
def human_expert_review(
    sku: SKU,
    ai_classification: AIClassification,
    expert: HumanExpert,
) -> HumanReview:
    """Simulate an expert review, including decision and elapsed review time."""
    review_time = expert.average_review_time
    if ai_classification.confidence_score < 0.7:
        review_time *= 1.3
    if len(ai_classification.uncertainty_factors) > 2:
        review_time *= 1.2
    review_time *= np.random.normal(1.0, 0.15)
    review_time = max(10.0, review_time)

    if ai_classification.confidence_score > 0.85:
        acceptance_probability = 0.9 * expert.trust_in_ai
    elif ai_classification.confidence_score > 0.7:
        acceptance_probability = 0.7 * expert.trust_in_ai
    else:
        acceptance_probability = 0.4 * expert.trust_in_ai

    has_domain_match = any(
        area.casefold() in sku.business_context.casefold()
        for area in expert.expertise_areas
    )
    if has_domain_match:
        acceptance_probability *= 0.8

    if np.random.random() < acceptance_probability:
        decision_status = DecisionStatus.HUMAN_ACCEPTED
        final_category = ai_classification.predicted_category
        reasoning = (
            f"The simulated expert accepted the AI recommendation with "
            f"ensemble vote share {ai_classification.confidence_score:.2f}. "
            f"{ai_classification.explanation}"
        )
        confidence_factors = [
            "AI vote share",
            "Classification explanation",
            "Simulated expert review",
        ]
    else:
        decision_status = DecisionStatus.HUMAN_OVERRIDDEN
        alternatives = list(ai_classification.alternative_categories)
        if alternatives and np.random.random() < 0.7:
            final_category = str(np.random.choice(alternatives))
        else:
            final_category = generate_expert_classification(sku, expert)
        reasoning = generate_expert_reasoning(
            sku, ai_classification, final_category, expert
        )
        confidence_factors = [
            "Domain knowledge represented by the simulation",
            "Business context",
            "Strategic considerations",
        ]

    considerations = generate_business_considerations(sku, expert)
    return HumanReview(
        reviewer_id=expert.expert_id,
        review_timestamp=time.strftime("%Y-%m-%d %H:%M:%S"),
        decision_status=decision_status,
        final_category=final_category,
        reasoning=reasoning,
        confidence_factors=confidence_factors,
        business_considerations=considerations,
        review_duration_seconds=review_time,
    )


def generate_expert_classification(sku: SKU, expert: HumanExpert) -> str:
    """Generate a rule-based simulated expert alternative category."""
    if sku.annual_value > 150000:
        abc = "A"
    elif sku.annual_value > 30000:
        abc = "B"
    else:
        abc = "C"

    if sku.cv < 0.12:
        xyz = "X"
    elif sku.cv < 0.30:
        xyz = "Y"
    else:
        xyz = "Z"

    context = sku.business_context.casefold()
    if "Electronics" in expert.expertise_areas and "electronics" in context:
        if sku.annual_value > 100000 and sku.cv > 0.15:
            xyz = "Y"
    if "Fashion" in expert.expertise_areas and "seasonal" in context:
        if sku.cv > 0.2:
            xyz = "Y"

    return abc + xyz


def generate_expert_reasoning(
    sku: SKU,
    ai_classification: AIClassification,
    expert_category: str,
    expert: HumanExpert,
) -> str:
    """Build a readable explanation for a simulated override."""
    reasoning_parts = [
        f"The AI recommended {ai_classification.predicted_category} with "
        f"ensemble vote share {ai_classification.confidence_score:.2f}."
    ]
    if "Electronics" in expert.expertise_areas:
        reasoning_parts.append(
            "The simulation considers technology lifecycle and obsolescence context."
        )
    elif "Fashion" in expert.expertise_areas:
        reasoning_parts.append(
            "The simulation considers seasonal patterns and trend cycles."
        )
    elif "Industrial equipment" in expert.expertise_areas:
        reasoning_parts.append(
            "The simulation considers product lifecycles and spare-parts context."
        )

    context = sku.business_context.casefold()
    if "new product" in context:
        reasoning_parts.append(
            "The business context identifies a new product with limited history."
        )
    elif "end-of-life" in context:
        reasoning_parts.append(
            "The business context identifies an end-of-life product phase."
        )
    if sku.strategic_importance == "CRITICAL":
        reasoning_parts.append(
            "The item is marked critical in the simulated business context."
        )
    reasoning_parts.append(
        f"The simulated expert assigns category {expert_category}."
    )
    return " ".join(reasoning_parts)


def generate_business_considerations(sku: SKU, expert: HumanExpert) -> List[str]:
    """Collect the contextual factors included in the simulated review."""
    considerations = []
    context = sku.business_context.casefold()
    if sku.market_trend:
        considerations.append(f"Market trend: {sku.market_trend}")
    if "seasonal" in context:
        considerations.append("Seasonal context is present")
    if sku.strategic_importance == "CRITICAL":
        considerations.append("The item is marked as strategically critical")
    elif sku.strategic_importance == "HIGH":
        considerations.append("The item is marked as strategically high importance")
    if sku.annual_value > 100000:
        considerations.append("Annual value exceeds the high-value threshold")
    if "Electronics" in expert.expertise_areas:
        considerations.append("The expert profile includes electronics expertise")
    return considerations


print("Simulated human review and reasoning functions are ready.")

In [ ]:
def human_ai_collaborative_classification(
    skus: List[SKU],
    ai_model: Dict,
    experts: List[HumanExpert],
    review_capacity: int = 3000,
) -> List[CollaborativeResult]:
    """Classify all SKUs, prioritize reviews, and apply the review capacity."""
    if review_capacity < 0:
        raise ValueError("review_capacity must be non-negative")
    if not experts and review_capacity > 0:
        raise ValueError("At least one expert is required when reviews are enabled")

    print(f"Starting simulated classification for {len(skus)} SKUs...")
    results = []
    review_queue = []

    print("Phase 1: Generate an ensemble recommendation for every SKU.")
    for index, sku in enumerate(skus, start=1):
        start_time = time.perf_counter()
        ai_classification = ai_classify_sku(sku, ai_model)
        ai_execution_time = time.perf_counter() - start_time
        priority = determine_review_priority(sku, ai_classification)
        if priority in ("HIGH", "MEDIUM"):
            review_queue.append(
                (sku, ai_classification, priority, ai_execution_time)
            )
        else:
            results.append(
                CollaborativeResult(
                    sku=sku,
                    ai_classification=ai_classification,
                    human_review=None,
                    final_category=ai_classification.predicted_category,
                    collaboration_type="AI_ONLY",
                    total_processing_time=ai_execution_time,
                    trust_score=ai_classification.confidence_score,
                )
            )
        if index % 5000 == 0:
            print(f"  Classified {index}/{len(skus)} SKUs.")

    review_queue.sort(
        key=lambda item: (item[2] != "HIGH", item[2] != "MEDIUM")
    )
    selected_reviews = review_queue[:review_capacity]
    capacity_deferred = review_queue[review_capacity:]
    print(
        f"Phase 2: {len(review_queue)} SKUs entered the review queue; "
        f"capacity permits {len(selected_reviews)} reviews."
    )

    # Deferred items keep their AI recommendation, so every input SKU appears
    # exactly once in the returned results.
    for sku, ai_classification, _, ai_execution_time in capacity_deferred:
        results.append(
            CollaborativeResult(
                sku=sku,
                ai_classification=ai_classification,
                human_review=None,
                final_category=ai_classification.predicted_category,
                collaboration_type="AI_ONLY",
                total_processing_time=ai_execution_time,
                trust_score=ai_classification.confidence_score,
            )
        )

    print("Phase 3: Review the selected queue with simulated experts.")
    for index, (sku, ai_classification, _, ai_execution_time) in enumerate(
        selected_reviews, start=1
    ):
        expert = select_expert_for_review(sku, experts)
        review_start_time = time.perf_counter()
        human_review = human_expert_review(sku, ai_classification, expert)
        review_execution_time = time.perf_counter() - review_start_time
        if human_review.decision_status == DecisionStatus.HUMAN_ACCEPTED:
            trust_score = (ai_classification.confidence_score + expert.trust_in_ai) / 2
            collaboration_type = "HUMAN_ACCEPTED"
        else:
            trust_score = expert.trust_in_ai * 0.7
            collaboration_type = "HUMAN_OVERRIDDEN"

        results.append(
            CollaborativeResult(
                sku=sku,
                ai_classification=ai_classification,
                human_review=human_review,
                final_category=human_review.final_category,
                collaboration_type=collaboration_type,
                total_processing_time=ai_execution_time + review_execution_time,
                trust_score=trust_score,
            )
        )
        if index % 1000 == 0:
            print(f"  Completed {index}/{len(selected_reviews)} human reviews.")

    if len(results) != len(skus):
        raise AssertionError("Each input SKU must have exactly one result")
    print("Collaborative classification completed.")
    return results


def calculate_partnership_metrics(
    results: List[CollaborativeResult],
) -> PartnershipMetrics:
    """Calculate descriptive workflow metrics without claiming accuracy."""
    total_skus = len(results)
    ai_only_count = sum(
        result.collaboration_type == "AI_ONLY" for result in results
    )
    human_accepted_count = sum(
        result.collaboration_type == "HUMAN_ACCEPTED" for result in results
    )
    human_overridden_count = sum(
        result.collaboration_type == "HUMAN_OVERRIDDEN" for result in results
    )
    human_reviewed_count = human_accepted_count + human_overridden_count
    reviewed_trust = [
        result.trust_score for result in results if result.human_review is not None
    ]

    return PartnershipMetrics(
        total_skus_processed=total_skus,
        ai_only_count=ai_only_count,
        human_reviewed_count=human_reviewed_count,
        human_overridden_count=human_overridden_count,
        collaborative_refined_count=sum(
            result.collaboration_type == "COLLABORATIVE" for result in results
        ),
        average_processing_time=(
            float(np.mean([result.total_processing_time for result in results]))
            if results
            else 0.0
        ),
        trust_development_score=(
            float(np.mean(reviewed_trust)) if reviewed_trust else float("nan")
        ),
        review_rate=human_reviewed_count / total_skus if total_skus else 0.0,
        override_rate=(
            human_overridden_count / human_reviewed_count
            if human_reviewed_count
            else 0.0
        ),
    )


print("The complete triage, review, and reporting workflow is ready.")

In [ ]:
# Generate the synthetic SKU population using the fixed seed established above.
print("Generating a synthetic dataset for the human-AI workflow...")

business_contexts = [
    "Electronics - Consumer devices",
    "Electronics - Computer components",
    "Electronics - Mobile accessories",
    "Fashion - Seasonal clothing",
    "Fashion - Trend items",
    "Fashion - Basic apparel",
    "Industrial - Manufacturing equipment",
    "Industrial - Spare parts",
    "Industrial - Tools",
    "Consumer goods - Fast moving",
    "Consumer goods - Household items",
    "Consumer goods - Personal care",
    "New product launch",
    "End-of-life phase",
    "Mature product",
    "Growth phase",
]
market_trends = [
    "Growing demand",
    "Stable market",
    "Declining trend",
    "Seasonal peak",
    "Volatile market",
    "Emerging trend",
]
strategic_levels = ["CRITICAL", "HIGH", "MEDIUM", "LOW"]
category_probabilities = [0.04, 0.03, 0.01, 0.08, 0.06, 0.04, 0.35, 0.25, 0.14]
category_names = [
    "high_stable", "high_moderate", "high_volatile",
    "medium_stable", "medium_moderate", "medium_volatile",
    "low_stable", "low_moderate", "low_volatile",
]

collaborative_skus = []
for index in range(25000):
    category = str(np.random.choice(category_names, p=category_probabilities))
    if category.startswith("high"):
        annual_value = np.random.uniform(80000, 600000)
        base_demand = np.random.uniform(60, 180)
    elif category.startswith("medium"):
        annual_value = np.random.uniform(20000, 80000)
        base_demand = np.random.uniform(20, 80)
    else:
        annual_value = np.random.uniform(5000, 25000)
        base_demand = np.random.uniform(5, 30)

    if "stable" in category:
        cv_target = np.random.uniform(0.02, 0.08)
    elif "moderate" in category:
        cv_target = np.random.uniform(0.08, 0.25)
    else:
        cv_target = np.random.uniform(0.25, 1.2)

    monthly_std = base_demand * cv_target
    monthly_demands = np.random.normal(
        base_demand, monthly_std, 12
    ).clip(0, None).tolist()

    collaborative_skus.append(
        SKU(
            sku_id=f"SKU_{index + 5000:05d}",
            annual_value=annual_value,
            monthly_demands=monthly_demands,
            business_context=str(np.random.choice(business_contexts)),
            strategic_importance=str(
                np.random.choice(
                    strategic_levels, p=[0.1, 0.2, 0.4, 0.3]
                )
            ),
            market_trend=str(np.random.choice(market_trends)),
        )
    )

values = np.array([sku.annual_value for sku in collaborative_skus])
cvs = np.array([sku.cv for sku in collaborative_skus])
print(f"Created {len(collaborative_skus):,} synthetic SKUs.")
print(f"Annual value range: ${values.min():,.0f} to ${values.max():,.0f}")
print(f"Mean annual value: ${values.mean():,.0f}")
print(f"Demand CV range: {cvs.min():.3f} to {cvs.max():.3f}")
print(f"Mean demand CV: {cvs.mean():.3f}")

for label, attribute, choices in [
    ("Business context", "business_context", business_contexts),
    ("Strategic importance", "strategic_importance", strategic_levels),
    ("Market trend", "market_trend", market_trends),
]:
    counts = {}
    for sku in collaborative_skus:
        value = getattr(sku, attribute)
        counts[value] = counts.get(value, 0) + 1
    print(f"\n{label} distribution:")
    for value in choices:
        count = counts.get(value, 0)
        print(f"  {value}: {count:,} ({count / len(collaborative_skus):.1%})")

In [ ]:
print("Running the simulated human-AI classification workflow")
print("=" * 68)

ai_model = create_ai_ensemble_classifier()
experts = create_human_experts()
REVIEW_CAPACITY = 3000

print(f"SKUs: {len(collaborative_skus):,}")
print(f"Human review capacity: {REVIEW_CAPACITY:,}")
print(f"Ensemble members: {len(ai_model['models'])}")
print(f"Expert profiles: {len(experts)}")
print("\nExpert profiles:")
for expert in experts:
    print(
        f"  {expert.name} ({expert.expert_id}): "
        f"{expert.expertise_areas[0]}, {expert.experience_years} years of experience"
    )

run_start = time.time()
collaborative_results = human_ai_collaborative_classification(
    skus=collaborative_skus,
    ai_model=ai_model,
    experts=experts,
    review_capacity=REVIEW_CAPACITY,
)
run_elapsed = time.time() - run_start
metrics = calculate_partnership_metrics(collaborative_results)

assert len(collaborative_results) == len(collaborative_skus)
assert sum(
    result.human_review is not None for result in collaborative_results
) == metrics.human_reviewed_count

print(f"\nWorkflow elapsed time: {run_elapsed:.2f} seconds")
print("\nDescriptive workflow results")
print(f"  SKUs with a final result: {metrics.total_skus_processed:,}")
print(f"  AI-only decisions: {metrics.ai_only_count:,}")
print(f"  Human-reviewed decisions: {metrics.human_reviewed_count:,}")
print(f"  Review rate: {metrics.review_rate:.1%}")
print(f"  Human overrides: {metrics.human_overridden_count:,}")
print(f"  Override rate among reviewed SKUs: {metrics.override_rate:.1%}")
print(f"  Mean Python execution time per SKU: {metrics.average_processing_time:.6f} seconds")
if np.isfinite(metrics.trust_development_score):
    print(f"  Mean trust score among reviewed SKUs: {metrics.trust_development_score:.3f}")
else:
    print("  Mean trust score among reviewed SKUs: not available")

category_counts = {}
for result in collaborative_results:
    category_counts[result.final_category] = (
        category_counts.get(result.final_category, 0) + 1
    )
print("\nFinal category counts:")
for category in sorted(category_counts):
    print(f"  {category}: {category_counts[category]:,}")

print("\nThese are simulation outputs, not measured accuracy or business-impact results.")

In [ ]:
# Visualize workflow behavior rather than claiming predictive performance.
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle("Simulated Human-AI Classification Workflow", fontsize=15)

collaboration_labels = ["AI only", "Human accepted", "Human overrode"]
collaboration_counts = [
    metrics.ai_only_count,
    metrics.human_reviewed_count - metrics.human_overridden_count,
    metrics.human_overridden_count,
]
axes[0, 0].bar(
    collaboration_labels,
    collaboration_counts,
    color=["steelblue", "seagreen", "darkorange"],
)
axes[0, 0].set_title("Decision pathway counts")
axes[0, 0].set_ylabel("Number of SKUs")
axes[0, 0].tick_params(axis="x", rotation=20)

final_categories = sorted(category_counts)
axes[0, 1].bar(
    final_categories,
    [category_counts[category] for category in final_categories],
    color="slateblue",
)
axes[0, 1].set_title("Final ABC/XYZ category counts")
axes[0, 1].set_xlabel("Category")
axes[0, 1].set_ylabel("Number of SKUs")

expert_names = [expert.name.split()[0] for expert in experts]
expert_accuracy_settings = [expert.accuracy_rate for expert in experts]
expert_ai_trust = [expert.trust_in_ai for expert in experts]
positions = np.arange(len(expert_names))
bar_width = 0.36
axes[0, 2].bar(
    positions - bar_width / 2,
    expert_accuracy_settings,
    bar_width,
    label="Profile accuracy setting",
)
axes[0, 2].bar(
    positions + bar_width / 2,
    expert_ai_trust,
    bar_width,
    label="Trust in AI setting",
)
axes[0, 2].set_title("Illustrative expert profile parameters")
axes[0, 2].set_ylabel("Configured score")
axes[0, 2].set_xticks(positions)
axes[0, 2].set_xticklabels(expert_names)
axes[0, 2].set_ylim(0, 1.05)
axes[0, 2].legend()

ai_confidence = [
    result.ai_classification.confidence_score for result in collaborative_results
]
review_trust = [
    result.trust_score
    for result in collaborative_results
    if result.human_review is not None
]
axes[1, 0].hist(
    ai_confidence, bins=np.linspace(0, 1, 21), alpha=0.7, label="AI vote share"
)
if review_trust:
    axes[1, 0].hist(
        review_trust,
        bins=np.linspace(0, 1, 21),
        alpha=0.7,
        label="Review trust score",
    )
axes[1, 0].set_title("Vote-share and review-trust distributions")
axes[1, 0].set_xlabel("Score")
axes[1, 0].set_ylabel("Number of SKUs")
axes[1, 0].legend()

ai_only_times = [
    result.total_processing_time
    for result in collaborative_results
    if result.human_review is None
]
human_reviewed_execution_times = [
    result.total_processing_time
    for result in collaborative_results
    if result.human_review is not None
]
human_times = [
    result.human_review.review_duration_seconds
    for result in collaborative_results
    if result.human_review is not None
]
time_series = []
time_labels = []
if ai_only_times:
    time_series.append(ai_only_times)
    time_labels.append("AI-only Python execution")
if human_reviewed_execution_times:
    time_series.append(human_reviewed_execution_times)
    time_labels.append("Reviewed Python execution")
if human_times:
    time_series.append(human_times)
    time_labels.append("Simulated human review")
if time_series:
    axes[1, 1].boxplot(time_series, tick_labels=time_labels, showfliers=False)
axes[1, 1].set_title("Execution and simulated review-time distributions")
axes[1, 1].set_ylabel("Seconds")
axes[1, 1].tick_params(axis="x", rotation=15)

axes[1, 2].bar(
    ["Reviewed", "Capacity"],
    [metrics.human_reviewed_count, REVIEW_CAPACITY],
    color=["teal", "lightgray"],
)
axes[1, 2].set_title("Review workload and configured capacity")
axes[1, 2].set_ylabel("Number of SKUs")

for axis in axes.flat:
    axis.grid(axis="y", alpha=0.25)

fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()
print("Workflow visualizations are displayed; profile settings are not performance measurements.")

In [ ]:
print("DETAILED WORKFLOW ANALYSIS")
print("=" * 68)

expert_by_id = {expert.expert_id: expert for expert in experts}
specialized_reviews = {expert.expert_id: [] for expert in experts}
for result in collaborative_results:
    if result.human_review is None:
        continue
    expert_id = result.human_review.reviewer_id
    expert = expert_by_id[expert_id]
    context = result.sku.business_context.casefold()
    if any(area.casefold() in context for area in expert.expertise_areas):
        specialized_reviews[expert_id].append(result)

print("\nReviews matched to an expert's stated context:")
for expert in experts:
    reviews = specialized_reviews[expert.expert_id]
    if reviews:
        override_count = sum(
            result.collaboration_type == "HUMAN_OVERRIDDEN" for result in reviews
        )
        print(f"  {expert.name}: {len(reviews):,} context-matched reviews")
        print(f"    Overrides in this subset: {override_count / len(reviews):.1%}")
    else:
        print(f"  {expert.name}: no context-matched reviews in this run")

print("\nHuman decisions by ensemble vote-share band:")
confidence_bands = [
    ("High (> 0.8)", lambda value: value > 0.8),
    ("Medium (0.6 to 0.8)", lambda value: 0.6 < value <= 0.8),
    ("Low (<= 0.6)", lambda value: value <= 0.6),
]
for label, condition in confidence_bands:
    band_results = [
        result
        for result in collaborative_results
        if result.human_review is not None
        and condition(result.ai_classification.confidence_score)
    ]
    if not band_results:
        print(f"  {label}: no reviewed SKUs")
        continue
    accepted = sum(
        result.collaboration_type == "HUMAN_ACCEPTED" for result in band_results
    )
    mean_trust = np.mean([result.trust_score for result in band_results])
    print(f"  {label}: {len(band_results):,} reviews")
    print(f"    Acceptance rate: {accepted / len(band_results):.1%}")
    print(f"    Mean simulated trust score: {mean_trust:.3f}")

review_durations = [
    result.human_review.review_duration_seconds
    for result in collaborative_results
    if result.human_review is not None
]
print("\nReview workload summary:")
print(f"  Configured review capacity: {REVIEW_CAPACITY:,}")
print(f"  Completed human reviews: {metrics.human_reviewed_count:,}")
print(f"  Mean simulated review duration: {np.mean(review_durations):.2f} seconds" if review_durations else "  Mean simulated review duration: not available")

print("\nValidation limits:")
print("  The synthetic dataset has no independently labeled ABC/XYZ target categories.")
print("  Therefore, accuracy, override correctness, and learning improvement are not measured.")
print("  The expert profile accuracy settings are inputs to the example, not observed outcomes.")
print("  No cost, service-level, stockout, or return-on-investment calculation is supported by these data.")


## Summary and conclusions

This notebook implements a simulated human–AI workflow for ABC/XYZ inventory classification. Its sequence is explicit: generate synthetic SKU data, obtain weighted ensemble votes, assign review priorities, apply a finite review capacity, simulate expert decisions, and summarize the resulting workflow.

The AI base category is determined by annual value and sample demand CV. The ensemble perturbs some simulated votes, and the winning weighted vote share is reported as a confidence score. Human review can accept or override the ensemble recommendation; context, strategic importance, and expert profiles influence parts of the simulation. Items that enter the review queue but fall beyond capacity keep their AI recommendation, ensuring that every input SKU receives exactly one final result.

The reported counts, vote shares, trust scores, and execution times characterize this simulation only. Measured Python execution time and simulated human review duration are distinct quantities. The notebook does not provide labeled outcomes, so it cannot establish classification accuracy or prove that human review improves decisions. It also does not calculate inventory savings, stockout reduction, productivity improvement, or learning gains. Those conclusions would require appropriate observed data, explicit outcome definitions, and validation beyond this workflow.

### Key concepts

- **ABC/XYZ classification:** A two-dimensional categorization using value and relative demand variability.
- **Ensemble vote share:** The fraction of total model weight supporting the winning category; it is not automatically a calibrated probability.
- **Human-in-the-loop triage:** Rules direct selected cases to review, while a capacity limit determines how many can be reviewed.
- **Simulation versus evidence:** Simulated decisions and configured profile parameters help illustrate workflow mechanics, but are not empirical measurements.

A practical extension would require business-specific value and variability thresholds, representative historical data, independently established target labels, and clearly defined measures of operational impact. Those additions are outside the scope of this simulation.